In [1]:
import torch, torch.nn as nn

# 1. Corpus & Simple Vocabulary
corpus = ["the movie was fantastic", "the movie was boring", "i loved the acting", "i hated the plot", "the plot was amazing", "the acting was terrible"]
words = sorted(list(set(" ".join(corpus).split())))
w2i = {w: i+1 for i, w in enumerate(words)} | {"<pad>": 0}
i2w = {i: w for w, i in w2i.items()}
vocab_size, max_len = len(w2i), max(len(line.split()) for line in corpus)

# 2. Build N-grams & Padding
X, y = [], []
for line in corpus:
    tokens = [w2i[w] for w in line.split()]
    for i in range(1, len(tokens)):
        seq = [0] * (max_len - len(tokens[:i+1])) + tokens[:i+1] # Left-padding
        X.append(seq[:-1])
        y.append(seq[-1])

X_train, y_train = torch.tensor(X), torch.tensor(y)

# 3. Clean Vanilla RNN Model
class TextModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 32)
        self.rnn = nn.RNN(32, 32, batch_first=True) # Straightforward vanilla RNN
        self.fc = nn.Linear(32, vocab_size)
        
    def forward(self, x):
        _, h = self.rnn(self.embed(x))
        return self.fc(h[-1]) # h[-1] automatically grabs the final hidden state

# 4. Training
model = TextModel(vocab_size)
optimizer, criterion = torch.optim.Adam(model.parameters(), lr=0.01), nn.CrossEntropyLoss()

for epoch in range(300):
    optimizer.zero_grad()
    loss = criterion(model(X_train), y_train)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 100 == 0: print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 5. Simple Predict Function
def predict(seed):
    tokens = [w2i.get(w, 0) for w in seed.lower().split()]
    padded = [0] * (max_len - 1 - len(tokens)) + tokens
    with torch.no_grad():
        pred = model(torch.tensor([padded])).argmax(1).item()
    return i2w.get(pred, "<unknown>")

print("the movie was ->", predict("the movie was"))
print("i loved the ->", predict("i loved the"))

Epoch 100, Loss: 0.3877
Epoch 200, Loss: 0.3863
Epoch 300, Loss: 0.3858
the movie was -> boring
i loved the -> acting
